[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvanCR/dl-semana6-Cunyas-Ramos/blob/main/notebooks/colab/02_lenet_vgg_colab.ipynb)

**Versión para Google Colab** (usar GPU). La primera celda clona el repositorio y descarga el dataset.

# 02 - Tarea 1: LeNet-5 y VGG-11 (con/sin Batch Normalization)
Entrenamiento desde cero de 4 variantes con **3 semillas** (42, 43, 44) cada una, para reportar media ± desviación. Config: Adam, lr=1e-3, batch=64, 20 épocas, entrada 64×64. El split train/val es fijo; solo cambia la semilla de inicialización, barajado y augmentation. Se restaura el checkpoint con mejor accuracy de validación y con él se evalúa en TEST.

In [ ]:
# --- Configuración para Google Colab ---
import os
REPO = 'dl-semana6-Cunyas-Ramos'
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://github.com/lvanCR/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!pip -q install kagglehub
!bash data/download_data.sh
%cd notebooks
import torch
assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import sys, json; sys.path.append('../src')
import pandas as pd  # importar pandas ANTES que torch: evita un crash del kernel en Windows
import numpy as np, torch, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix
from utils import get_loaders, FIG_DIR, ROOT, CLASSES
from models import build_model
from train import run_experiment

RES_DIR = ROOT/'results'; CKPT_DIR = RES_DIR/'checkpoints'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
EPOCHS, LR, BS, SEEDS = 20, 1e-3, 64, [42, 43, 44]
train_loader, val_loader, test_loader = get_loaders(img_size=64, batch_size=BS)

## Entrenamiento: 4 variantes × 3 semillas

In [ ]:
VARIANTS = {'lenet': ('lenet', False), 'lenet_bn': ('lenet', True),
            'vgg11': ('vgg11', False), 'vgg11_bn': ('vgg11', True)}
runs = {t: [] for t in VARIANTS}
for tag, (name, bn) in VARIANTS.items():
    for s in SEEDS:
        r = run_experiment(lambda: build_model(name, bn), tag, train_loader, val_loader, test_loader,
                           EPOCHS, LR, device, s, CKPT_DIR)
        runs[tag].append(r)
        print(f"{tag:9s} seed {s} | params {r['params']:,} | best val {max(r['hist']['val_acc']):.4f} | test {r['test_acc']:.4f}")

## Tabla resumen (media ± desviación sobre 3 semillas)

In [ ]:
def first_epoch_over(h, thr=0.8):
    return next((i+1 for i, a in enumerate(h['val_acc']) if a >= thr), np.nan)

ms = lambda v, d=4: f"{np.mean(v):.{d}f} ± {np.std(v):.{d}f}"
rows = []
for t, rs in runs.items():
    rows.append({'Modelo': t, 'Parámetros': rs[0]['params'],
                 'Tiempo/época (s)': ms([np.mean(r['hist']['epoch_time'][1:]) for r in rs], 2),
                 'Mejor val acc': ms([max(r['hist']['val_acc']) for r in rs]),
                 'Test acc': ms([r['test_acc'] for r in rs]),
                 'Épocas a 80% val': ms([first_epoch_over(r['hist']) for r in rs], 1)})
summary = pd.DataFrame(rows); summary.to_csv(RES_DIR/'t1_resumen.csv', index=False)
json.dump({t: [dict(seed=r['seed'], params=r['params'], hist=r['hist'], test_acc=r['test_acc'],
                    test_loss=r['test_loss'], y_true=r['y_true'], y_pred=r['y_pred']) for r in rs]
           for t, rs in runs.items()}, open(RES_DIR/'t1_resultados.json', 'w'))
summary

In [ ]:
# detalle por semilla
pd.DataFrame([{'Modelo': t, 'Semilla': r['seed'], 'Mejor val acc': round(max(r['hist']['val_acc']), 4),
               'Test acc': round(r['test_acc'], 4)} for t, rs in runs.items() for r in rs]).pivot(
    index='Modelo', columns='Semilla', values=['Mejor val acc', 'Test acc'])

## Curvas de pérdida y accuracy (media ± desviación entre semillas)

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 7), sharex=True)
for j, (t, rs) in enumerate(runs.items()):
    ep = np.arange(1, EPOCHS+1)
    for i, (k, lab) in enumerate([('loss', 'Cross-entropy'), ('acc', 'Accuracy')]):
        for split, col in [('train', 'tab:blue'), ('val', 'tab:orange')]:
            a = np.array([r['hist'][f'{split}_{k}'] for r in rs]); m, s = a.mean(0), a.std(0)
            axs[i, j].plot(ep, m, color=col, label=split); axs[i, j].fill_between(ep, m-s, m+s, color=col, alpha=.2)
        axs[i, j].set_title(f"{t} - {'pérdida' if k == 'loss' else 'accuracy'}"); axs[i, j].set_ylabel(lab)
        axs[i, j].legend()
    axs[1, j].set_xlabel('Época')
fig.suptitle('Tarea 1: curvas de entrenamiento y validación (3 semillas)'); plt.tight_layout()
plt.savefig(FIG_DIR/'t1_curvas.png', dpi=150); plt.show()

## Diagnóstico de la brecha validación → test
Matriz de confusión en TEST (semilla 42) y accuracy por clase, para ver qué clases fallan.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (t, rs) in zip(axs, runs.items()):
    r = rs[0]; cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=[c[:4] for c in CLASSES], yticklabels=[c[:4] for c in CLASSES])
    ax.set_title(f"{t} (test acc {r['test_acc']:.3f})"); ax.set_xlabel('Predicha'); ax.set_ylabel('Real')
fig.suptitle('Matrices de confusión en TEST (semilla 42)'); plt.tight_layout()
plt.savefig(FIG_DIR/'t1_confusion_test.png', dpi=150); plt.show()
pd.DataFrame({t: np.diag(confusion_matrix(rs[0]['y_true'], rs[0]['y_pred'], normalize='true')).round(3)
              for t, rs in runs.items()}, index=CLASSES)